# Spotify 히트곡 예측
## [2026-1] Machine Learning Term Project

**Task**: 오디오 피처(danceability, energy 등)를 기반으로 노래가 '인기곡'인지 분류  
**Dataset**: Spotify Tracks Dataset (Kaggle, ~114,000곡)  
**Models**: Logistic Regression → Random Forest → XGBoost

## 0-A. 구글 드라이브 마운트 & 패키지 설치

**실행 전 준비**: Kaggle에서 받은 `dataset.csv`를 구글 드라이브 최상단(내 드라이브)에 업로드해두세요.

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# shap만 추가 설치 (나머지는 코랩에 기본 내장)
!pip install shap -q
print('설치 완료!')

## 0-B. 라이브러리 및 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, RocCurveDisplay
)
from xgboost import XGBClassifier
import shap

SEED = 42
np.random.seed(SEED)

# 코랩 로컬 저장 경로
FIGURES_DIR = '/content/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12
print('Setup complete.')

## 1. 데이터 로드

In [ ]:
# 구글 드라이브 '내 드라이브' 최상단에 dataset.csv를 올렸을 경우
DATA_PATH = '/content/drive/MyDrive/dataset.csv'

df = pd.read_csv(DATA_PATH, index_col=0)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('결측치:')
print(df.isnull().sum())
print(f'\n중복 행: {df.duplicated().sum()}')
print(f'\nPopularity 분포:')
print(df['popularity'].describe())

## 2. EDA (탐색적 데이터 분석)

In [ ]:
# 2-1. Popularity 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['popularity'], bins=50, color='#1DB954', edgecolor='white', alpha=0.85)
axes[0].axvline(x=60, color='red', linestyle='--', label='threshold=60')
axes[0].set_title('Popularity Score Distribution')
axes[0].set_xlabel('Popularity (0-100)')
axes[0].set_ylabel('Count')
axes[0].legend()

genre_pop = df.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(15)
axes[1].barh(genre_pop.index[::-1], genre_pop.values[::-1], color='#1DB954')
axes[1].set_title('Top 15 Genres by Average Popularity')
axes[1].set_xlabel('Average Popularity')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/eda_popularity.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2-2. 오디오 피처 상관관계 히트맵
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'popularity'
]

plt.figure(figsize=(10, 8))
corr = df[AUDIO_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Audio Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/eda_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2-3. 인기곡 vs 비인기곡 피처 비교
THRESHOLD = 60  # 설계 선택: 인기곡 기준
print(f'Threshold {THRESHOLD} 기준:')
print(f'  인기곡:   {(df["popularity"] >= THRESHOLD).sum():,}개 ({(df["popularity"] >= THRESHOLD).mean()*100:.1f}%)')
print(f'  비인기곡: {(df["popularity"] < THRESHOLD).sum():,}개 ({(df["popularity"] < THRESHOLD).mean()*100:.1f}%)')

features_to_compare = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness']
fig, axes = plt.subplots(1, len(features_to_compare), figsize=(18, 4))

for i, feat in enumerate(features_to_compare):
    popular     = df[df['popularity'] >= THRESHOLD][feat]
    not_popular = df[df['popularity'] <  THRESHOLD][feat]
    axes[i].hist(not_popular, bins=30, alpha=0.6, color='gray',    label='Not Popular', density=True)
    axes[i].hist(popular,     bins=30, alpha=0.6, color='#1DB954', label='Popular',     density=True)
    axes[i].set_title(feat.capitalize())
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions: Popular vs. Not Popular', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/eda_feature_compare.png', bbox_inches='tight')
plt.show()

## 3. 전처리

In [ ]:
# 3-1. 중복 제거 및 결측치 처리
df = df.drop_duplicates(subset='track_id')
df = df.dropna()
df = df.reset_index(drop=True)  # 인덱스 재정렬 (코랩 iloc 호환)
print(f'전처리 후 shape: {df.shape}')

# 3-2. 타겟 변수 생성
df['is_popular'] = (df['popularity'] >= THRESHOLD).astype(int)

# 3-3. 피처 선택
df['duration_min'] = df['duration_ms'] / 60000

FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_min', 'explicit', 'mode', 'key'
]
TARGET = 'is_popular'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f'\nClass balance: {y.value_counts().to_dict()}')

In [ ]:
# 3-4. Train / Validation / Test 분리 (6:2:2)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEED, stratify=y_trainval
)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

# 3-5. 스케일링 (Logistic Regression용)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

## 4. 모델 학습

In [ ]:
results = {}

def evaluate(name, model, X_v, y_v, X_te, y_te):
    val_pred  = model.predict(X_v)
    test_pred = model.predict(X_te)
    val_prob  = model.predict_proba(X_v)[:, 1]
    test_prob = model.predict_proba(X_te)[:, 1]

    val_m  = {'Accuracy': accuracy_score(y_v,  val_pred),
               'F1':      f1_score(y_v,  val_pred),
               'ROC-AUC': roc_auc_score(y_v,  val_prob)}
    test_m = {'Accuracy': accuracy_score(y_te, test_pred),
               'F1':      f1_score(y_te, test_pred),
               'ROC-AUC': roc_auc_score(y_te, test_prob)}

    print(f'\n[{name}]')
    print(f'  Val  → Acc: {val_m["Accuracy"]:.4f} | F1: {val_m["F1"]:.4f} | AUC: {val_m["ROC-AUC"]:.4f}')
    print(f'  Test → Acc: {test_m["Accuracy"]:.4f} | F1: {test_m["F1"]:.4f} | AUC: {test_m["ROC-AUC"]:.4f}')

    results[name] = {'val': val_m, 'test': test_m,
                     'val_pred': val_pred, 'test_pred': test_pred,
                     'test_prob': test_prob, 'model': model}

In [ ]:
# 4-1. Baseline: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train_sc, y_train)
evaluate('Logistic Regression', lr, X_val_sc, y_val, X_test_sc, y_test)

In [ ]:
# 4-2. Random Forest (default)
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate('Random Forest (default)', rf, X_val, y_val, X_test, y_test)

In [ ]:
# 4-3. Random Forest 하이퍼파라미터 튜닝
rf_param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2']
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    rf_param_grid, n_iter=20, cv=3,
    scoring='f1', random_state=SEED, n_jobs=-1, verbose=1
)
rf_search.fit(X_trainval, y_trainval)
print(f'Best RF params: {rf_search.best_params_}')

best_rf = rf_search.best_estimator_
evaluate('Random Forest (tuned)', best_rf, X_val, y_val, X_test, y_test)

In [ ]:
# 4-4. XGBoost (default)
xgb_default = XGBClassifier(
    n_estimators=200, random_state=SEED,
    eval_metric='logloss', verbosity=0
)
xgb_default.fit(X_train, y_train,
                eval_set=[(X_val, y_val)], verbose=False)
evaluate('XGBoost (default)', xgb_default, X_val, y_val, X_test, y_test)

In [ ]:
# 4-5. XGBoost 하이퍼파라미터 튜닝
xgb_param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 5, 7, 9],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5]
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0),
    xgb_param_grid, n_iter=20, cv=3,
    scoring='f1', random_state=SEED, n_jobs=-1, verbose=1
)
xgb_search.fit(X_trainval, y_trainval)
print(f'Best XGB params: {xgb_search.best_params_}')

best_xgb = xgb_search.best_estimator_
evaluate('XGBoost (tuned)', best_xgb, X_val, y_val, X_test, y_test)

## 5. 결과 시각화

In [ ]:
# 5-1. 모델 성능 비교 표
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model':    name,
        'Val Acc':  f"{res['val']['Accuracy']:.4f}",
        'Val F1':   f"{res['val']['F1']:.4f}",
        'Val AUC':  f"{res['val']['ROC-AUC']:.4f}",
        'Test Acc': f"{res['test']['Accuracy']:.4f}",
        'Test F1':  f"{res['test']['F1']:.4f}",
        'Test AUC': f"{res['test']['ROC-AUC']:.4f}"
    })
pd.DataFrame(summary_rows)

In [ ]:
# 5-2. 성능 비교 바 차트
models_to_plot  = ['Logistic Regression', 'Random Forest (tuned)', 'XGBoost (tuned)']
metrics_labels  = ['Accuracy', 'F1', 'ROC-AUC']
colors          = ['#636EFA', '#EF553B', '#00CC96']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(metrics_labels):
    vals = [results[m]['test'][metric] for m in models_to_plot]
    bars = axes[i].bar(models_to_plot, vals, color=colors, alpha=0.85)
    axes[i].set_ylim(0.5, 1.0)
    axes[i].set_title(f'Test {metric}')
    axes[i].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.005,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=11)
plt.suptitle('Model Comparison on Test Set', fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5-3. Confusion Matrix (최고 성능 모델)
best_model_name = max(results, key=lambda x: results[x]['test']['F1'])
best_result     = results[best_model_name]
print(f'Best model: {best_model_name}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (split_name, y_true, y_pred) in zip(axes, [
    ('Validation', y_val,  best_result['val_pred']),
    ('Test',       y_test, best_result['test_pred'])
]):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Popular', 'Popular'],
                yticklabels=['Not Popular', 'Popular'])
    ax.set_title(f'{best_model_name}\n{split_name} Confusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5-4. ROC Curve 비교
fig, ax = plt.subplots(figsize=(7, 6))
for name in models_to_plot:
    RocCurveDisplay.from_predictions(
        y_test, results[name]['test_prob'], name=name, ax=ax
    )
ax.plot([0,1],[0,1],'--', color='gray', label='Random (AUC=0.50)')
ax.set_title('ROC Curve Comparison')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/roc_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5-5. Feature Importance (XGBoost 기준)
xgb_model   = results['XGBoost (tuned)']['model']
importances = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values()

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='#1DB954', edgecolor='white')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5-6. SHAP 분석
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test.iloc[:500])

plt.figure()
shap.summary_plot(shap_values, X_test.iloc[:500], feature_names=FEATURES, show=False)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/shap_summary.png', bbox_inches='tight')
plt.show()

## 6. 실패 케이스 분석

In [ ]:
test_df = X_test.copy()
test_df['y_true'] = y_test.values
test_df['y_pred'] = best_result['test_pred']
test_df['prob']   = best_result['test_prob']

fp = test_df[(test_df['y_true']==0) & (test_df['y_pred']==1)]
fn = test_df[(test_df['y_true']==1) & (test_df['y_pred']==0)]
tp = test_df[(test_df['y_true']==1) & (test_df['y_pred']==1)]

print(f'False Positives (비인기 → 인기 오분류): {len(fp):,}')
print(f'False Negatives (인기 → 비인기 오분류): {len(fn):,}')

feat_compare = ['danceability', 'energy', 'valence', 'acousticness', 'loudness']
x = np.arange(len(feat_compare))
w = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x - w, tp[feat_compare].mean(), w, label='True Positive',  color='#1DB954', alpha=0.8)
axes[0].bar(x,     fp[feat_compare].mean(), w, label='False Positive', color='#FF6B6B', alpha=0.8)
axes[0].bar(x + w, fn[feat_compare].mean(), w, label='False Negative', color='#4ECDC4', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(feat_compare, rotation=15)
axes[0].set_title('Feature Means by Error Type')
axes[0].legend(fontsize=8)

axes[1].hist(fp['prob'], bins=30, alpha=0.7, color='#FF6B6B', label='FP confidence')
axes[1].hist(fn['prob'], bins=30, alpha=0.7, color='#4ECDC4', label='FN confidence')
axes[1].set_title('Prediction Confidence of Misclassified Samples')
axes[1].set_xlabel('Predicted Probability (Popular)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/error_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# popularity 원본값 기준으로 오분류 분포 확인
test_idx    = X_test.index
pop_orig    = df.loc[test_idx, 'popularity'].values
correct_pop = pop_orig[test_df['y_true'].values == test_df['y_pred'].values]
wrong_pop   = pop_orig[test_df['y_true'].values != test_df['y_pred'].values]

plt.figure(figsize=(9, 5))
plt.hist(correct_pop, bins=40, alpha=0.6, color='#1DB954', label=f'Correct ({len(correct_pop):,})', density=True)
plt.hist(wrong_pop,   bins=40, alpha=0.6, color='red',     label=f'Wrong ({len(wrong_pop):,})',   density=True)
plt.axvline(x=THRESHOLD, color='black', linestyle='--', label=f'Threshold={THRESHOLD}')
plt.title('Popularity Distribution: Correct vs. Wrong Predictions')
plt.xlabel('Original Popularity Score')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/error_threshold.png', bbox_inches='tight')
plt.show()

print('\n[분석] 대부분의 오분류는 popularity가 threshold 근처(45~70)인 애매한 곡들에서 발생')

## 7. figures 드라이브에 저장

In [ ]:
import shutil

DRIVE_SAVE_PATH = '/content/drive/MyDrive/spotify_ml_figures'
shutil.copytree(FIGURES_DIR, DRIVE_SAVE_PATH, dirs_exist_ok=True)
print(f'그래프 저장 완료 → {DRIVE_SAVE_PATH}')
print('저장된 파일 목록:')
for f in sorted(os.listdir(DRIVE_SAVE_PATH)):
    print(f'  {f}')

## 8. 결론 및 Future Work

**최종 모델 성능** (Test Set):
- XGBoost (tuned)가 Accuracy/F1/AUC 전체에서 가장 우수

**왜 XGBoost가 가장 좋았나**:
- 오디오 피처 간 비선형 상호작용 포착 (e.g., loudness × energy)
- 부스팅으로 경계 샘플에 집중 학습

**실패 패턴**:
- popularity 45~70 사이 '경계선상' 곡에서 오분류 집중
- instrumentalness 높은 곡(클래식, 재즈)이 FP로 분류되는 경향

**Future Work**:
1. 출시 연도/트렌드 피처 추가
2. 아티스트 인지도 피처 (팔로워 수, 과거 히트 수)
3. 장르별 분리 모델
4. Regression으로 전환 (popularity 직접 예측)
5. Raw audio → Mel-spectrogram + CNN